# Experiments
This file will evaluate the results of the synthetic dataset with the ground truth

## Object Detection


In [ ]:
import numpy as np
import trimesh

def obb_iou_3d(corners1, corners2):
    """
    Compute IoU between two oriented 3D bounding boxes represented by 8 corner points.

    Parameters:
        corners1, corners2: np.ndarray of shape (8,3)
    
    Returns:
        iou: float
    """
    # Create convex hull meshes
    hull1 = trimesh.convex.convex_hull(corners1)
    hull2 = trimesh.convex.convex_hull(corners2)

    # Compute volumes
    vol1 = hull1.volume
    vol2 = hull2.volume

    # Compute intersection volume
    try:
        inter = hull1.intersection(hull2)
        inter_vol = inter.volume if inter.is_volume else 0.0
    except Exception:
        inter_vol = 0.0

    # Compute IoU
    union_vol = vol1 + vol2 - inter_vol
    iou = inter_vol / union_vol if union_vol > 0 else 0.0
    return iou

def evaluate_obb(pred_boxes, gt_boxes, iou_threshold=0.5):
    """
    Evaluate lists of oriented 3D bounding boxes (8 points each).

    Parameters:
        pred_boxes: list of np.ndarray of shape (8,3)
        gt_boxes: list of np.ndarray of shape (8,3)
        iou_threshold: float
    
    Returns:
        precision, recall, iou_matches
    """
    matched_gt = set()
    iou_matches = []
    tp = 0

    for pb in pred_boxes:
        best_iou = 0
        best_gt_idx = -1
        for i, gb in enumerate(gt_boxes):
            if i in matched_gt:
                continue
            iou = obb_iou_3d(pb, gb)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = i
        if best_iou >= iou_threshold:
            tp += 1
            matched_gt.add(best_gt_idx)
            iou_matches.append(best_iou)

    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    return precision, recall, iou_matches



In [ ]:
jsonFile = ""
detectionFile =""

In [ ]:
# Example usage
# Each box is 8 corners (numpy array of shape (8,3))
gt_box = np.array([[0,0,0],[2,0,0],[2,2,0],[0,2,0],[0,0,2],[2,0,2],[2,2,2],[0,2,2]])
pred_box = np.array([[1,1,1],[3,1,1],[3,3,1],[1,3,1],[1,1,3],[3,1,3],[3,3,3],[1,3,3]])

precision, recall, iou_matches = evaluate_obb([pred_box], [gt_box])
print("Precision:", precision)
print("Recall:", recall)
print("IoU matches:", iou_matches)